# 3. Adjacency Matrix Builder (Optional)

This notebook builds CSR (Compressed Sparse Row) adjacency matrices from the chunked edge lists.
This is useful for certain graph sampling algorithms or if you need efficient neighbor lookups outside of PyG's default loaders.

In [ ]:
# Configuration
import os

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
LMDB_NODE_DIR = os.path.join(ROOT_DIR, "lmdb_node_mapping")
CHUNKS_DIR = os.path.join(ROOT_DIR, "processed_data", "chunks")
ADJACENCY_DIR = os.path.join(ROOT_DIR, "processed_data", "adjacency")

# Node Mapping Paths (for counts)
NODE_LMDB_PATHS = {
    "pekerja": os.path.join(LMDB_NODE_DIR, "pekerja.lmdb"),
    "nasabah": os.path.join(LMDB_NODE_DIR, "nasabah.lmdb"),
    "simpanan": os.path.join(LMDB_NODE_DIR, "simpanan.lmdb"),
    "pinjaman": os.path.join(LMDB_NODE_DIR, "pinjaman.lmdb"),
    "transaksi": os.path.join(LMDB_NODE_DIR, "transaksi.lmdb"),
}

# Edge configurations to build Adjacency for
EDGES_CONFIG = [
    {
        "name": "nasabah_pekerja",
        "src_type": "nasabah",
        "dst_type": "pekerja",
        "chunks_path": os.path.join(CHUNKS_DIR, "edges_edge_nasabah_is_pekerja")
    },
    # Add others as needed
]

os.makedirs(ADJACENCY_DIR, exist_ok=True)

In [ ]:
# Imports
import numpy as np
import torch
import lmdb
from tqdm.notebook import tqdm
import glob
import scipy.sparse as sp

In [ ]:
def get_num_nodes(lmdb_path):
    if not os.path.exists(lmdb_path): return 0
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    with env.begin() as txn:
        return txn.stat()['entries']

def build_csr_from_pt_chunks(src_nodes, dst_nodes, chunks_path, out_name):
    if not os.path.exists(chunks_path):
        print(f"Chunks not found: {chunks_path}")
        return
        
    files = glob.glob(os.path.join(chunks_path, "*.pt"))
    
    # Pass 1: Count degrees for Indptr
    # We use numpy memmap to handle large arrays if needed, 
    # but for indptr (size=num_nodes) it usually fits in RAM.
    
    # Using int32 for indptr is usually safe up to 2B edges, 
    # use int64 if you expect more.
    degree = np.zeros(src_nodes + 1, dtype=np.int64)
    
    print(f"Pass 1: Counting degrees for {out_name}...")
    total_edges = 0
    for f in tqdm(files):
        chunk = torch.load(f, weights_only=True)
        src = chunk[0].numpy()
        # fast histogram
        # bins must enable counting up to src_nodes
        counts = np.bincount(src, minlength=src_nodes + 1)
        # pad if counts is smaller than degree arrays (unlikely with minlength)
        l = len(counts)
        if l > len(degree):
             # This means we found IDs larger than src_nodes! Error in mapping?
             # Or estimation was wrong.
             # Resize degree
             new_degree = np.zeros(l, dtype=np.int64)
             new_degree[:len(degree)] = degree
             degree = new_degree
             
        degree[:l] += counts
        total_edges += chunk.size(1)

    # Build indptr
    indptr = np.zeros(len(degree), dtype=np.int64)
    np.cumsum(degree, out=indptr)
    
    # Pass 2: Fill Indices
    print(f"Pass 2: Filling indices (Total edges: {total_edges})...")
    
    # Output arrays
    indices_path = os.path.join(ADJACENCY_DIR, f"{out_name}_indices.npy")
    indptr_path = os.path.join(ADJACENCY_DIR, f"{out_name}_indptr.npy")
    
    # Memory map indices array
    indices = np.memmap(indices_path, dtype=np.int32, mode='w+', shape=(total_edges,))
    
    # Temporary counter to track insertion position per node
    # We copy indptr to use as current_pointer
    current_ptr = indptr[:-1].copy()
    
    for f in tqdm(files):
        chunk = torch.load(f, weights_only=True)
        src = chunk[0].numpy()
        dst = chunk[1].numpy()
        
        for s, d in zip(src, dst):
            pos = current_ptr[s]
            indices[pos] = d
            current_ptr[s] += 1
            
    # Save indptr
    np.save(indptr_path, indptr)
    
    # Flush memmap
    indices.flush()
    del indices
    
    print(f"Saved CSR to {ADJACENCY_DIR}")

In [ ]:
for cfg in EDGES_CONFIG:
    src_n = get_num_nodes(NODE_LMDB_PATHS[cfg["src_type"]])
    dst_n = get_num_nodes(NODE_LMDB_PATHS[cfg["dst_type"]])
    build_csr_from_pt_chunks(src_n, dst_n, cfg["chunks_path"], cfg["name"])